# Session 2: friction loss, Reynolds number and pump power
**A. Real-world problem.** A pump must push water through a long pipe. A thin pipe is cheap
but the water rubs against the wall and the pump must work hard; a fat pipe costs more but
wastes less energy. How do we put a number on the friction, and find the smallest pipe that
keeps the pump within its power budget?

**Plan.** (1) Reynolds number: is the flow smooth or turbulent?, (2) friction factor,
(3) Darcy-Weisbach head loss plus minor losses, (4) pump power, (5) design: choose the
smallest standard diameter within the budget. Educational model, not a design tool.

In [ ]:
import math
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

RHO = 998.0      # water density at 20 C, kg/m^3
NU = 1.0e-6      # kinematic viscosity of water at 20 C, m^2/s
G = 9.81         # gravity, m/s^2

## Step 1: Reynolds number
Re = v D / nu compares the push of the moving water (inertia) with its stickiness
(viscosity). Below about 2000 the flow is smooth layers (laminar); above about 4000 it is
churning (turbulent); in between nobody can predict it well.

In [ ]:
def velocity(q, d):
    """Average speed (m/s) for flow q (m^3/s) in a round pipe of inside diameter d (m)."""
    return 4.0 * q / (math.pi * d ** 2)


def reynolds(q, d, nu=NU):
    """Reynolds number (no unit)."""
    return velocity(q, d) * d / nu

## Step 2: friction factor f
* Laminar (Re < 2000): f = 64 / Re, exact.
* Turbulent (Re > 4000): the Swamee-Jain formula, with relative roughness e/D
  (e = height of the wall bumps):  f = 0.25 / [log10( e/(3.7 D) + 5.74 / Re^0.9 )]^2
* Between 2000 and 4000 we simply join the two with a straight line (a modelling choice).

`colebrook` is the older, slower formula that needs iteration; it is our reference.

In [ ]:
def friction_factor(re, rel_rough):
    """Darcy friction factor (no unit) for Reynolds number re and relative roughness e/D."""
    def turbulent(r):
        return 0.25 / math.log10(rel_rough / 3.7 + 5.74 / r ** 0.9) ** 2
    if re < 2000.0:
        return 64.0 / re
    if re > 4000.0:
        return turbulent(re)
    w = (re - 2000.0) / 2000.0
    return (1 - w) * (64.0 / 2000.0) + w * turbulent(4000.0)


def colebrook(re, rel_rough, iterations=60):
    """Reference turbulent friction factor from the Colebrook-White equation (fixed-point iteration)."""
    f = 0.02
    for _ in range(iterations):
        f = 1.0 / (-2.0 * math.log10(rel_rough / 3.7 + 2.51 / (re * math.sqrt(f)))) ** 2
    return f

## Step 3: head loss
Head is energy per unit weight of water, written as a height in metres. Friction along the
pipe (Darcy-Weisbach) and bends, valves and entrances ("minor losses", K is their total
loss coefficient) each cost head:

    h_f = f (L / D) v^2 / (2 g)         h_m = K v^2 / (2 g)

In [ ]:
def head_loss(q, d, length, rough, k_minor=0.0, nu=NU):
    """Total friction + minor head loss (m). rough = wall roughness e in metres."""
    v = velocity(q, d)
    f = friction_factor(reynolds(q, d, nu), rough / d)
    return (f * length / d + k_minor) * v ** 2 / (2 * G)

## Step 4: pump power
The pump lifts the water by the static rise `lift` and also pays the head loss:
H = lift + h_loss. Water power = rho g Q H (watts). A real pump and motor waste some,
so the electric power is water power divided by the efficiency eta.

In [ ]:
def pump_head(q, d, length, rough, lift, k_minor=0.0, nu=NU):
    return lift + head_loss(q, d, length, rough, k_minor, nu)


def pump_power(q, d, length, rough, lift, k_minor=0.0, eta=0.65, nu=NU):
    """Electric power (W) the pump needs."""
    return RHO * G * q * pump_head(q, d, length, rough, lift, k_minor, nu) / eta

## Step 5: design
Pipes come in standard sizes. The power falls steeply as the diameter grows (v falls, so
v^2 falls, and the friction term has 1/D on top). The smallest diameter inside the budget
is usually the cheapest pipe that works. We also keep v below 3 m/s (noise, wear).

In [ ]:
STANDARD_MM = [50, 65, 80, 100, 125, 150, 200, 250, 300]


def smallest_diameter(budget_w, q, length, rough, lift, k_minor=0.0, eta=0.65, vmax=3.0, sizes_mm=STANDARD_MM):
    """Smallest standard diameter (mm) with electric power <= budget and speed <= vmax, or None."""
    for mm in sizes_mm:
        d = mm / 1000.0
        if velocity(q, d) <= vmax and pump_power(q, d, length, rough, lift, k_minor, eta) <= budget_w:
            return mm
    return None

## Step 6: see it

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    q, length, rough, lift, k, eta = 0.020, 500.0, 0.00015, 10.0, 5.0, 0.65
    budget = 7000.0
    print(f"{'D (mm)':>7} {'v (m/s)':>8} {'Re':>9} {'f':>7} {'h_loss (m)':>11} {'power (kW)':>11}")
    for mm in STANDARD_MM:
        d = mm / 1000.0
        print(f"{mm:7d} {velocity(q, d):8.2f} {reynolds(q, d):9.0f} "
              f"{friction_factor(reynolds(q, d), rough / d):7.4f} {head_loss(q, d, length, rough, k):11.2f} "
              f"{pump_power(q, d, length, rough, lift, k, eta) / 1e3:11.2f}")
    print("smallest diameter within", budget / 1e3, "kW:", smallest_diameter(budget, q, length, rough, lift, k, eta), "mm")

    dd = np.linspace(0.06, 0.30, 100)
    pw = [pump_power(q, d, length, rough, lift, k, eta) / 1e3 for d in dd]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(dd * 1000, pw, label="pump electric power")
    ax.axhline(budget / 1e3, color="tab:red", ls="--", label="budget 7 kW")
    ax.set_ylim(0, 30)
    ax.set_xlabel("pipe inside diameter (mm)")
    ax.set_ylabel("pump electric power (kW)")
    ax.set_title("Power falls steeply as the pipe grows (20 L/s, 500 m)")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "power_vs_diameter.png", dpi=120)

    res = np.logspace(2.5, 7, 200)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    for rr in (1e-5, 1e-3, 1e-2):
        ax2.loglog(res, [friction_factor(r, rr) for r in res], label=f"e/D = {rr:g}")
    ax2.set_xlabel("Reynolds number (no unit)")
    ax2.set_ylabel("friction factor f (no unit)")
    ax2.set_title("Friction factor: laminar line, transition, turbulent curves")
    ax2.grid(alpha=0.3, which="both")
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "friction_factor.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Print the power for every standard size at 25 L/s instead of 20 L/s (keep the other numbers). Which size is the smallest within 9 kW?
2. **Stretch.** Add a function `speed_limit_diameter(q, vmax)` that returns the smallest standard size whose speed is at most vmax, and use it to explain why 65 mm fails the 3 m/s limit at 20 L/s.
3. **Explain.** Explain, using the table the lesson prints, why the best pipe size depends on the budget and on the speed limit and not only on the flow rate.

## Assessment
- **A. Conceptual.** Why does doubling the pipe diameter reduce the friction loss by much more than a factor of two for the same flow rate?
- **B. Numerical.** 12 L/s of water flows through 200 m of 100 mm pipe of roughness 0.15 mm. Find the speed, Reynolds number, friction factor, head loss and the power lost to friction (no minor losses).
- **C. Graph and simulation interpretation.** Read the power-against-diameter curve for 20 L/s, 500 m, 10 m lift. About what power is needed at 100 mm, 125 mm and 200 mm, and what limits the curve for large diameters?
- **D. Design and optimisation challenge.** Design: the same job (20 L/s, 500 m, 10 m lift, K = 5, roughness 0.15 mm, 65 percent efficient) but a smaller budget of 5 kW. Choose a standard size from 100, 125, 150, 200 mm. What if the budget were 4 kW? Justify with speed and trade-offs.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math

import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    q, d, length, e, k = 0.020, 0.125, 500.0, 0.00015, 5.0
    v = L.velocity(q, d)
    re = L.reynolds(q, d)
    f = L.friction_factor(re, e / d)
    checks.append(("worked example: 20 L/s in 125 mm: v = 1.630 m/s, Re = 203700",
                   abs(v - 1.6297) < 1e-3 and abs(re - 203718) < 10))
    checks.append(("worked example: f = 0.0219, friction loss 11.87 m, minor loss 0.68 m",
                   abs(f - 0.0219) < 5e-5 and abs(f * length / d * v ** 2 / 19.62 - 11.87) < 0.02
                   and abs(k * v ** 2 / 19.62 - 0.68) < 0.01))
    h = L.pump_head(q, d, length, e, 10.0, k)
    p = L.pump_power(q, d, length, e, 10.0, k, 0.65)
    checks.append((f"worked example: head 22.55 m, water power 4.42 kW, electric 6.79 kW (got {p / 1e3:.2f})",
                   abs(h - 22.55) < 0.01 and abs(998 * 9.81 * q * h - 4415) < 5 and abs(p - 6793) < 5))
    checks.append(("design: smallest standard size within 7 kW is 125 mm",
                   L.smallest_diameter(7000.0, q, length, e, 10.0, k, 0.65) == 125))
    checks.append(("design: 100 mm needs more than 14 kW", L.pump_power(q, 0.100, length, e, 10.0, k) > 14000))
    checks.append(("an impossible budget returns None", L.smallest_diameter(100.0, q, length, e, 10.0, k) is None))

    # hand calculation, laminar: Hagen-Poiseuille h = 128 nu L Q / (pi g D^4)
    ql, dl = 2e-5, 0.025
    hp = 128 * L.NU * 10.0 * ql / (math.pi * L.G * dl ** 4)
    checks.append(("laminar flow (Re = 1019): f = 64/Re and loss equals Hagen-Poiseuille",
                   L.reynolds(ql, dl) < 2000 and abs(L.head_loss(ql, dl, 10.0, 0.0) - hp) < 1e-9))

    # Swamee-Jain against the Colebrook reference
    worst = 0.0
    for re_ in (5e3, 2e4, 1e5, 1e6, 1e7):
        for rr in (1e-5, 1e-4, 1e-3, 1e-2):
            worst = max(worst, abs(L.friction_factor(re_, rr) / L.colebrook(re_, rr) - 1))
    checks.append((f"Swamee-Jain is within 3% of Colebrook for Re 5000-1e7 (worst {100 * worst:.1f}%)", worst < 0.03))

    # limits: continuity at the regime joins, scaling laws
    j1 = abs(L.friction_factor(1999.999, 1e-3) - L.friction_factor(2000.001, 1e-3))
    j2 = abs(L.friction_factor(3999.999, 1e-3) - L.friction_factor(4000.001, 1e-3))
    checks.append(("friction factor is continuous at Re = 2000 and 4000", j1 < 1e-4 and j2 < 1e-4))
    a = L.head_loss(0.010, 0.1, 100.0, 0.0001)
    b = L.head_loss(0.020, 0.1, 100.0, 0.0001)
    checks.append(("doubling the flow roughly quadruples the loss (3.7 to 4.1 times)", 3.7 < b / a < 4.1))
    checks.append(("smoother pipe gives less loss", L.head_loss(q, d, length, 0.0) < L.head_loss(q, d, length, 0.001)))
    checks.append(("zero length and zero K give zero loss", L.head_loss(q, d, 0.0, e, 0.0) == 0.0))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
